## Final Pipeline

**Goal:** Build one final reusable scikit-learn pipeline and generate a Kaggle submission file.


The final pipeline includes:

1. Missing value imputation.
2. One-hot encoding.
3. Scaling.
4. PCA.
5. Ridge Regression.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor

pd.set_option('display.max_columns', 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'reports'
MODELS = PROJECT_ROOT / 'models'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

def make_one_hot_encoder():
    """Compatibility helper for different scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)



train = pd.read_csv(r'C:\Users\divin\OneDrive\Desktop\Data_Science\project\UK_House_Price_Prediction_Model\train_features.csv')
test = pd.read_csv(r'C:\Users\divin\OneDrive\Desktop\Data_Science\project\UK_House_Price_Prediction_Model\test_features.csv')
print('Loaded train_features:', train.shape)
print('Loaded test_features:', test.shape)

Loaded train_features: (1460, 90)
Loaded test_features: (1459, 89)


#### Prepare final training data

For the final pipeline, we train on the full training dataset because we already evaluated the approach in Notebook 04.

In [5]:
y = np.log1p(train['SalePrice'])
X = train.drop(columns=['SalePrice', 'Id'])

test_ids = test['Id']
X_test = test.drop(columns=['Id'])

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print('Training feature shape:', X.shape)
print('Test feature shape:', X_test.shape)
print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

Training feature shape: (1460, 88)
Test feature shape: (1459, 88)
Numeric features: 45
Categorical features: 43


#### Build the final preprocessing and PCA pipeline

Important: the test data must go through exactly the same transformations as the training data.

This is why `Pipeline` is important.

In [7]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', make_one_hot_encoder())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# These values come from the study/tuning step in Notebook 04.
# If your Notebook 04 finds different best values, update them here.
final_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('model', Ridge(alpha=20))
])

final_pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['MSSubClass', 'LotFrontage',
                                                   'LotArea', 'OverallQual',
                                                   'OverallCond', 'YearBuilt',
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                                   '1stFlrSF', '2ndFlrSF',
                                                   'LowQualFi...
                                                   'Neighborhood', 'Condition1',
                                                   'Condition2', 'BldgType',
                                                   'HouseStyle', 'RoofStyle',
                                                   'RoofMatl', 'Exterior1st',
                                                   'Exterior2nd', 'MasVnrType',
                                                   'ExterQual', 'ExterCond',
                                                   'Foundation', 'BsmtQual',
                                                   'BsmtCond', 'BsmtExposure',
                                                   'BsmtFinType1',
                                                   'BsmtFinType2', 'Heating',
                                                   'HeatingQC', 'CentralAir',
                                                   'Electrical', ...])])),
                ('pca', PCA(n_components=0.95, random_state=42)),
                ('model', Ridge(alpha=20))])

The model predicts `log1p(SalePrice)`, so we convert back using `np.expm1()`.

In [8]:
test_pred_log = final_pipeline.predict(X_test)
test_pred = np.expm1(test_pred_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})

submission_path = REPORTS / 'submission_pca.csv'
submission.to_csv(submission_path, index=False)

print('Saved submission to:', submission_path)
display(submission.head())

NotFittedError: This ColumnTransformer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

#### Final project explanation

You can explain the final model like this:

> I built a complete machine learning workflow to predict house sale prices. I first understood the dataset and identified feature types and missing-value meanings. I then performed EDA to study important relationships. After that, I created engineered features such as house age, total square footage, total bathrooms, and binary indicators for garage, basement, fireplace, and pool. I used a scikit-learn pipeline to handle missing values, encode categorical features, scale numeric features, apply PCA for dimensionality reduction, and train a Ridge Regression model. The final pipeline was saved and used to generate predictions for the Kaggle test set.

This satisfies the portfolio requirements:

- Data understanding and ETL.
- Feature engineering.
- Encoding and scaling.
- PCA.
- Model training and evaluation.
- Hyperparameter tuning.
- Cross-validation.
- Final scikit-learn pipeline.
- Communication of findings.